# Pretraining Dataset Preparation

## Objectives
By the end of this demo you will be able to:
1. **Source** text data for pretraining from the Hugging Face Hub (using `wikitext`) and from GitHub (Python scripts).
2. **Contrast** the structure of pretraining data (raw text) vs. fine-tuning data (instruction–response pairs).
3. **Apply** four standard data-cleaning steps used in real LLM pretraining pipelines:
   - Filter out too-short documents
   - Remove intra-document paragraph repetitions
   - Deduplicate the corpus
   - Remove non-English documents
4. **Save** the cleaned dataset to disk in Parquet format for downstream tokenisation.

## Description
Large Language Models are pretrained on *massive* corpora of raw text.
Before any text reaches the trainer it must pass through a **data preparation pipeline** that removes
low-quality, duplicate, or irrelevant content. This notebook walks through a minimal but realistic
version of that pipeline end-to-end.

**Dataset used:** `wikitext` (`wikitext-103-raw-v1`) — a standard Wikipedia-derived benchmark,
widely used in NLP research and fully available on the Hugging Face Hub.

In [1]:
import warnings
warnings.filterwarnings("ignore")


In [2]:
# Uncomment and run if packages are not already installed
!pip install datasets langdetect -q


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 981.5/981.5 kB 18.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done


## 1. Sourcing Datasets for Pretraining

There are two common ways to gather raw text for pretraining:
1. **Download an existing dataset** from the Hugging Face Hub.
2. **Scrape and compile** domain-specific text (e.g. Python code from GitHub).

 In production, corpora like *The Pile*, *C4*, or *RedPajama*
> (trillions of tokens) are used.

### 1a. Download a Text Dataset from Hugging Face

We use **WikiText-103** — a clean, encyclopaedic English corpus derived from Wikipedia

In [3]:
import datasets

# Load WikiText-103 pretraining dataset (raw, no tokenisation)
pretraining_dataset = datasets.load_dataset(
    "Salesforce/wikitext",
    name="wikitext-103-raw-v1",
    split="train"
)
print(pretraining_dataset)


README.md: 0.00B [00:00, ?B/s]

wikitext-103-raw-v1/test-00000-of-00001.(…):   0%|          | 0.00/733k [00:00<?, ?B/s]

wikitext-103-raw-v1/train-00000-of-00002(…):   0%|          | 0.00/157M [00:00<?, ?B/s]

wikitext-103-raw-v1/train-00001-of-00002(…):   0%|          | 0.00/157M [00:00<?, ?B/s]

wikitext-103-raw-v1/validation-00000-of-(…):   0%|          | 0.00/657k [00:00<?, ?B/s]

Generating test split:   0%|          | 0/4358 [00:00<?, ? examples/s]

Generating train split:   0%|          | 0/1801350 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/3760 [00:00<?, ? examples/s]

Dataset({
    features: ['text'],
    num_rows: 1801350
})


In [4]:
print("Columns:", pretraining_dataset.column_names)
print("\nSample text (first 500 chars):")
sample = next(x for x in pretraining_dataset if len(x["text"]) > 100)
print(sample["text"][:500])


Columns: ['text']

Sample text (first 500 chars):
 Senjō no Valkyria 3 : Unrecorded Chronicles ( Japanese : 戦場のヴァルキュリア3 , lit . Valkyria of the Battlefield 3 ) , commonly referred to as Valkyria Chronicles III outside Japan , is a tactical role @-@ playing video game developed by Sega and Media.Vision for the PlayStation Portable . Released in January 2011 in Japan , it is the third game in the Valkyria series . Employing the same fusion of tactical and real @-@ time gameplay as its predecessors , the story runs parallel to the first game and f


In [5]:
# Keep only the text column
pretraining_dataset = pretraining_dataset.select_columns(["text"])
print(pretraining_dataset)


Dataset({
    features: ['text'],
    num_rows: 1801350
})


#### Demo-size Subset
We work with **10,000 samples** to keep the notebook fast on any machine.
Remove `.select(...)` when running on a full cluster.


In [6]:
pretraining_dataset = pretraining_dataset.select(range(10_000))
print(f"Working with {pretraining_dataset.num_rows:,} samples")


Working with 10,000 samples


### 1b. Compare Pretraining vs Fine-tuning Data

We load an instruction-tuning dataset to contrast the structural difference.
Alpaca-GPT4: 52K instruction–response pairs, commonly used for SFT.


In [7]:
instruction_dataset = datasets.load_dataset(
    "c-s-ale/alpaca-gpt4-data",
    split="train"
)
print(instruction_dataset)


README.md: 0.00B [00:00, ?B/s]

data/alpaca_gpt4_data.json:   0%|          | 0.00/43.4M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/52002 [00:00<?, ? examples/s]

Dataset({
    features: ['instruction', 'input', 'output'],
    num_rows: 52002
})


In [8]:
i = 0
print(" Fine-tuning sample : - ")
print("Instruction : " + instruction_dataset[i]["instruction"])
print("Input       : " + instruction_dataset[i]["input"])
print("Output      : " + instruction_dataset[i]["output"])


 Fine-tuning sample : - 
Instruction : Give three tips for staying healthy.
Input       : 
Output      : 1. Eat a balanced and nutritious diet: Make sure your meals are inclusive of a variety of fruits and vegetables, lean protein, whole grains, and healthy fats. This helps to provide your body with the essential nutrients to function at its best and can help prevent chronic diseases.

2. Engage in regular physical activity: Exercise is crucial for maintaining strong bones, muscles, and cardiovascular health. Aim for at least 150 minutes of moderate aerobic exercise or 75 minutes of vigorous exercise each week.

3. Get enough sleep: Getting enough quality sleep is crucial for physical and mental well-being. It helps to regulate mood, improve cognitive function, and supports healthy growth and immune function. Aim for 7-9 hours of sleep each night.


> - **Pretraining data** = raw, unstructured text (Wikipedia, books, code…)
> - **Fine-tuning data** = structured instruction → response pairs
>
> Pretraining gives broad world knowledge; fine-tuning aligns behaviour.

### 1c. Scrape Python Code from GitHub

A real pretraining corpus blends prose, data, and **code**.
We fetch Python scripts from GitHub Raw URLs.


In [9]:
import os, requests

code_dir = "./code"
os.makedirs(code_dir, exist_ok=True)


In [10]:
urls = [
    "https://raw.githubusercontent.com/TheAlgorithms/Python/master/searches/double_linear_search_recursion.py",
    "https://raw.githubusercontent.com/TheAlgorithms/Python/master/sorts/bubble_sort.py",
    "https://raw.githubusercontent.com/TheAlgorithms/Python/master/maths/fibonacci.py",
    "https://raw.githubusercontent.com/TheAlgorithms/Python/master/data_structures/linked_list/singly_linked_list.py",
    "https://raw.githubusercontent.com/PaliC/pytorch/master/test/fx/test_subgraph_rewriter.py",
]


In [11]:
for url in urls:
    print(f"Fetching: {url.split('/')[-1]}")
    response = requests.get(url)
    file_path = os.path.join(code_dir, os.path.basename(url))
    with open(file_path, "wb") as f:
        f.write(response.content)

print("\nFiles saved:")
for file in os.listdir(code_dir):
    print(" -", file)


Fetching: double_linear_search_recursion.py
Fetching: bubble_sort.py
Fetching: fibonacci.py
Fetching: singly_linked_list.py
Fetching: test_subgraph_rewriter.py

Files saved:
 - double_linear_search_recursion.py
 - singly_linked_list.py
 - fibonacci.py
 - test_subgraph_rewriter.py
 - bubble_sort.py


In [12]:
code_dataset = []
for file in os.listdir(code_dir):
    with open(os.path.join(code_dir, file), "r", errors="ignore") as f:
        code_dataset.append({"text": f.read()})

code_dataset = datasets.Dataset.from_list(code_dataset)
print(code_dataset)


Dataset({
    features: ['text'],
    num_rows: 5
})


### 1d. Combine Text and Code Datasets

In [13]:
dataset = datasets.concatenate_datasets([pretraining_dataset, code_dataset])
print(f"Combined dataset: {dataset.num_rows:,} rows")
print(dataset)


Combined dataset: 10,005 rows
Dataset({
    features: ['text'],
    num_rows: 10005
})


## 2. Data Cleaning

| Step | What it removes |
|---|---|
| **Length filter** | Documents too short to carry useful signal |
| **Repetition filter** | Paragraphs excessively copy-pasted within a doc |
| **Deduplication** | Exact-duplicate documents across the corpus |
| **Language filter** | Non-English documents |


In [14]:
print(f"Rows before cleaning: {dataset.num_rows:,}")


Rows before cleaning: 10,005


### 2.1 Filter Short Documents

character-count filter.


In [15]:
def paragraph_length_filter(x):
    """
    Remove documents that are too short to carry useful signal.
    Uses character count — more robust than line count for datasets
    where each row is a sentence (e.g. WikiText).

    Threshold: keep documents with >= 50 characters.
    """
    return len(x["text"].strip()) >= 50


In [16]:
dataset = dataset.filter(paragraph_length_filter, load_from_cache_file=False)
print(f"After length filter: {dataset.num_rows:,} rows")


Filter:   0%|          | 0/10005 [00:00<?, ? examples/s]

After length filter: 4,422 rows


### 2.2 Remove Intra-document Repetitions

Thresholds:
- Remove if > 30% of paragraphs are duplicates OR
- Remove if duplicate characters exceed 20% of total document length


In [17]:
import re

def find_duplicates(paragraphs):
    """Count duplicate paragraphs and total duplicate characters."""
    unique_p = set()
    dup_chars = 0
    dup_count = 0
    for p in paragraphs:
        if p in unique_p:
            dup_chars += len(p)
            dup_count += 1
        else:
            unique_p.add(p)
    return dup_count, dup_chars

def paragraph_repetition_filter(x):
    """Return False if the document has too many repeated paragraphs."""
    text = x["text"]
    paragraphs = re.compile(r"\n{2,}").split(text.strip())
    if len(paragraphs) == 0:
        return False
    dup_paragraphs, dup_chars = find_duplicates(paragraphs)
    if dup_paragraphs / len(paragraphs) > 0.3:
        return False
    if dup_chars / len(text) > 0.2:
        return False
    return True


In [18]:
dataset = dataset.filter(paragraph_repetition_filter, load_from_cache_file=False)
print(f"After repetition filter: {dataset.num_rows:,} rows")


Filter:   0%|          | 0/4422 [00:00<?, ? examples/s]

After repetition filter: 4,421 rows


### 2.3 Deduplication

Exact-match deduplication using a seen-text set.

> For large-scale pipelines, MinHash/LSH handles near-duplicates.
> Exact dedup is the fast baseline.


In [19]:
def deduplication(ds):
    """Remove exact duplicate documents from a dataset."""
    seen = set()

    def is_unique(x):
        if x["text"] in seen:
            return False
        seen.add(x["text"])
        return True

    return ds.filter(is_unique, load_from_cache_file=False, num_proc=1)

dataset = deduplication(dataset)
print(f"After deduplication: {dataset.num_rows:,} rows")


Filter:   0%|          | 0/4421 [00:00<?, ? examples/s]

After deduplication: 4,419 rows


### 2.4 Quality Filter — Language Detection

Retain only English documents using `langdetect`.

> For production, fastText language ID (176 languages, >99% accuracy) is standard. `langdetect` is used here for simplicity — no model file download required.


In [20]:
def english_language_filter(x):
    """
    Keep documents that are predominantly ASCII — a reliable proxy for English.
    Short documents (< 100 chars) are kept by default.
    """
    text = x["text"].strip()
    if len(text) < 100:
        return True   # too short to judge — keep
    ascii_count = sum(1 for c in text if ord(c) < 128)
    return (ascii_count / len(text)) > 0.9


In [21]:
dataset = dataset.filter(english_language_filter, load_from_cache_file=False, num_proc=1)
print(f"After language filter: {dataset.num_rows:,} rows")


Filter:   0%|          | 0/4419 [00:00<?, ? examples/s]

After language filter: 4,418 rows


In [22]:
print(" Cleaning Pipeline Summary ")
print(f"Final clean dataset: {dataset.num_rows:,} rows")
print(dataset)


 Cleaning Pipeline Summary 
Final clean dataset: 4,418 rows
Dataset({
    features: ['text'],
    num_rows: 4418
})


## 3. Save the Dataset to Disk

Saving as Parquet — columnar binary format, faster than CSV, built-in compression, and the standard format for HuggingFace datasets.


In [23]:
os.makedirs("./data", exist_ok=True)
file_path = "./data/preprocessed_dataset.parquet"
dataset.to_parquet(file_path)
print(f"Saved to: {file_path}")


Creating parquet from Arrow format:   0%|          | 0/5 [00:00<?, ?ba/s]

Saved to: ./data/preprocessed_dataset.parquet


In [26]:
from google.colab import drive
drive.mount('/content/drive')

# Define the path to save the dataset in Google Drive
drive_data_dir = '/content/drive/MyDrive/Colab Notebooks/data'
os.makedirs(drive_data_dir, exist_ok=True)
file_path_drive = os.path.join(drive_data_dir, 'preprocessed_dataset.parquet')

dataset.to_parquet(file_path_drive)
print(f"Saved to: {file_path_drive}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


Creating parquet from Arrow format:   0%|          | 0/5 [00:00<?, ?ba/s]

Saved to: /content/drive/MyDrive/Colab Notebooks/data/preprocessed_dataset.parquet


In [30]:
# Reload later with:
#dataset = datasets.Dataset.from_parquet("./data/preprocessed_dataset.parquet")
dataset = datasets.Dataset.from_parquet("/content/drive/MyDrive/Colab Notebooks/data/preprocessed_dataset.parquet")

print(dataset)


Generating train split: 0 examples [00:00, ? examples/s]

Dataset({
    features: ['text'],
    num_rows: 4418
})


| Concept | Summary |
|---|---|
| Pretraining data | Raw, unstructured text — no labels needed |
| Fine-tuning data | Structured instruction–response pairs |
| Length filter | Removes trivially short/empty documents |
| Repetition filter | Removes copy-paste spam within a document |
| Deduplication | Removes identical documents across corpus |
| Language filter | Retains only the target language |
| Parquet format | Efficient binary format for large datasets |